In [ ]:
import pandas as pd
import torch
from collections import Counter

## Construir vocabulario desde los archivos CSV

In [ ]:
# Leer los archivos de entrenamiento y test para español
train_es = pd.read_csv('../fechas2/fechas2_train.es.csv')
test_es = pd.read_csv('../fechas2/fechas2_test.es.csv')

# Combinar todos los textos
all_texts = list(train_es['txt']) + list(test_es['txt'])

# Extraer todas las palabras
all_words = []
for text in all_texts:
    words = text.lower().split()
    all_words.extend(words)

# Contar frecuencias
word_counts = Counter(all_words)
print(f"Total palabras únicas: {len(word_counts)}")
print(f"\nPalabras más frecuentes:")
for word, count in word_counts.most_common(20):
    print(f"  {word}: {count}")

## Implementación del Tokenizador

In [ ]:
class Fechas2Tokenizer():
    """Tokenizador orientado a palabras para la tarea de fechas2.
    
    Similar a DigitSumTokenizer pero adaptado al vocabulario de fechas.
    """
    
    def __init__(self, vocab_file=None):
        """Inicializa el tokenizador.
        
        Args:
            vocab_file: Archivo CSV con los textos para construir vocabulario.
                       Si es None, usa un vocabulario predefinido.
        """
        if vocab_file is not None:
            self.build_vocab_from_file(vocab_file)
        else:
            self.build_default_vocab()
        
        self.index2word = {v: k for k, v in self.word2index.items()}
        self.vocab_size = len(self.word2index)
        print(f"Vocabulario construido con {self.vocab_size} palabras")
    
    def build_vocab_from_file(self, csv_file):
        """Construye vocabulario desde archivo CSV."""
        df = pd.read_csv(csv_file)
        
        # Extraer todas las palabras únicas
        words = set()
        for text in df['txt']:
            words.update(text.lower().split())
        
        # Ordenar alfabéticamente para consistencia
        words = sorted(list(words))
        
        # Crear diccionario de palabras a índices
        self.word2index = {}
        
        # Tokens especiales
        self.word2index['<pad>'] = 0
        self.word2index['<sos>'] = 1
        self.word2index['<eos>'] = 2
        
        # Añadir palabras del vocabulario
        for i, word in enumerate(words, start=3):
            self.word2index[word] = i
    
    def build_default_vocab(self):
        """Construye vocabulario predefinido con palabras comunes de fechas en español."""
        vocab_words = [
            # Días de la semana
            'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo',
            'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
            # Palabras comunes
            'el', 'la', 'los', 'las', 'un', 'una', 'en', 'de', 'por',
            'favor', 'gracias', 'please', 'thank', 'you', 'thanks',
            # Temporales
            'siguiente', 'próximo', 'próxima', 'este', 'esta', 'que', 'viene',
            'next', 'this', 'coming',
            # Días relativos
            'mañana', 'pasado', 'ayer', 'hoy',
            'tomorrow', 'day', 'after', 'yesterday', 'today', 'the',
            # Números y expresiones
            'días', 'día', 'days', 'tres', 'three', 'un', 'par', 'couple', 'of', 'a', 'in',
        ]
        
        self.word2index = {}
        self.word2index['<pad>'] = 0
        self.word2index['<sos>'] = 1
        self.word2index['<eos>'] = 2
        
        for i, word in enumerate(sorted(set(vocab_words)), start=3):
            self.word2index[word] = i
    
    def encode(self, text, seq_len=-1):
        """Codifica texto a secuencia de índices.
        
        Args:
            text: Texto a codificar
            seq_len: Longitud máxima de secuencia. Si > len(text), se rellena con <pad>
        
        Returns:
            Tensor con índices de tokens
        """
        words = text.lower().split()
        
        # Añadir tokens especiales
        tokens = [self.word2index['<sos>']]
        
        for word in words:
            if word in self.word2index:
                tokens.append(self.word2index[word])
            else:
                print(f"Advertencia: palabra '{word}' no está en el vocabulario")
                # Opción: ignorar la palabra o usar un token <unk>
        
        tokens.append(self.word2index['<eos>'])
        
        # Padding si es necesario
        if seq_len > len(tokens):
            tokens = tokens + [self.word2index['<pad>']] * (seq_len - len(tokens))
        
        return torch.tensor(tokens)
    
    def decode(self, indices):
        """Decodifica secuencia de índices a texto.
        
        Args:
            indices: Lista o tensor de índices
        
        Returns:
            Texto decodificado
        """
        if isinstance(indices, torch.Tensor):
            indices = indices.tolist()
        
        words = []
        for idx in indices:
            if idx in self.index2word:
                word = self.index2word[idx]
                # No incluir tokens especiales en la salida
                if word not in ['<pad>', '<sos>', '<eos>']:
                    words.append(word)
        
        return ' '.join(words)

## Crear tokenizador desde archivo de entrenamiento

In [ ]:
# Crear tokenizador con vocabulario del archivo de entrenamiento
tokenizer = Fechas2Tokenizer('../fechas2/fechas2_train.es.csv')

print(f"\nTamaño del vocabulario: {tokenizer.vocab_size}")
print(f"\nPrimeras 30 palabras del vocabulario:")
for i in range(min(30, tokenizer.vocab_size)):
    print(f"  {i}: {tokenizer.index2word[i]}")

## Probar el tokenizador

In [ ]:
# Ejemplos de codificación
ejemplos = [
    "por favor el siguiente jueves",
    "mañana",
    "pasado mañana",
    "el viernes que viene",
    "en tres días"
]

print("Pruebas de codificación/decodificación:\n")
for texto in ejemplos:
    encoded = tokenizer.encode(texto)
    decoded = tokenizer.decode(encoded)
    print(f"Original:    {texto}")
    print(f"Codificado:  {encoded.tolist()}")
    print(f"Decodificado: {decoded}")
    print()

## Probar con secuencia de longitud fija

In [ ]:
# Codificar con longitud fija
texto = "por favor el siguiente jueves"
seq_len = 15

encoded = tokenizer.encode(texto, seq_len=seq_len)
print(f"Texto: {texto}")
print(f"Codificado (seq_len={seq_len}): {encoded.tolist()}")
print(f"Longitud: {len(encoded)}")
print(f"Decodificado: {tokenizer.decode(encoded)}")

## Guardar el tokenizador para uso posterior

In [ ]:
import pickle

# Guardar tokenizador
with open('fechas2_tokenizer_es.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Tokenizador guardado en 'fechas2_tokenizer_es.pkl'")

## Crear tokenizador para inglés

In [ ]:
# Crear tokenizador para inglés
tokenizer_en = Fechas2Tokenizer('../fechas2/fechas2_train.en.csv')

print(f"\nTamaño del vocabulario (inglés): {tokenizer_en.vocab_size}")

# Guardar tokenizador
with open('fechas2_tokenizer_en.pkl', 'wb') as f:
    pickle.dump(tokenizer_en, f)

print("Tokenizador inglés guardado en 'fechas2_tokenizer_en.pkl'")

## Crear tokenizador bilingüe (español + inglés)

In [ ]:
class Fechas2BilingualTokenizer(Fechas2Tokenizer):
    """Tokenizador bilingüe para español e inglés."""
    
    def __init__(self, train_es_file, train_en_file):
        """Inicializa tokenizador bilingüe.
        
        Args:
            train_es_file: Archivo CSV con textos en español
            train_en_file: Archivo CSV con textos en inglés
        """
        # Leer ambos archivos
        df_es = pd.read_csv(train_es_file)
        df_en = pd.read_csv(train_en_file)
        
        # Extraer palabras únicas de ambos idiomas
        words = set()
        for text in list(df_es['txt']) + list(df_en['txt']):
            words.update(text.lower().split())
        
        # Ordenar alfabéticamente
        words = sorted(list(words))
        
        # Crear diccionario
        self.word2index = {
            '<pad>': 0,
            '<sos>': 1,
            '<eos>': 2,
        }
        
        for i, word in enumerate(words, start=3):
            self.word2index[word] = i
        
        self.index2word = {v: k for k, v in self.word2index.items()}
        self.vocab_size = len(self.word2index)
        print(f"Vocabulario bilingüe construido con {self.vocab_size} palabras")

# Crear tokenizador bilingüe
tokenizer_bilingual = Fechas2BilingualTokenizer(
    '../fechas2/fechas2_train.es.csv',
    '../fechas2/fechas2_train.en.csv'
)

# Guardar
with open('fechas2_tokenizer_bilingual.pkl', 'wb') as f:
    pickle.dump(tokenizer_bilingual, f)

print("Tokenizador bilingüe guardado en 'fechas2_tokenizer_bilingual.pkl'")

## Pruebas finales

In [ ]:
# Probar tokenizador bilingüe con ejemplos en ambos idiomas
ejemplos_bilingual = [
    "por favor el siguiente jueves",
    "please next thursday",
    "mañana",
    "tomorrow",
    "en tres días",
    "in three days"
]

print("Pruebas con tokenizador bilingüe:\n")
for texto in ejemplos_bilingual:
    encoded = tokenizer_bilingual.encode(texto)
    decoded = tokenizer_bilingual.decode(encoded)
    print(f"Original:     {texto}")
    print(f"Codificado:   {encoded.tolist()}")
    print(f"Decodificado: {decoded}")
    print()